# 06 — Snapshots, provenance and your own calendars

This notebook explains **where the data comes from**, why it is frozen, how an upstream
correction reaches you, and how to add your own closures without forking anything.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

import better_calendar as bcal
from better_calendar.calendars import snapshot

## 1. The problem snapshots solve

If `bcal.get("XNYS")` asked `exchange-calendars` at query time, then a
`pip install --upgrade` could move a settlement date with nobody deciding to. The back
office recalculates, the date shifts by a day, nobody saw it happen.

The fix: the data is materialised once, **committed**, shipped inside the wheel, and read
from disk. The providers are not even installed on the machines that consume the library.

In [2]:
# Three completely separate moments.
pd.DataFrame(
    [
        {"when": "rarely, by hand",
         "what": "better-calendar snapshot: imports the 4 upstreams, writes the files",
         "who": "you"},
        {"when": "at wheel build time",
         "what": "the committed files are copied — no computation at all",
         "who": "uv build"},
        {"when": "at query time",
         "what": "reads a local file, ~0.06 ms",
         "who": "the user"},
    ]
).set_index("when")

,what,who
when,,
"rarely, by hand",better-calendar snapshot: imports the 4 upstre...,you
at wheel build time,the committed files are copied — no computatio...,uv build
at query time,"reads a local file, ~0.06 ms",the user


In [3]:
# Proof: no provider is imported, even after querying everything.
code = (
    "import sys, better_calendar as bcal\n"
    "[bcal.get(n) for n in bcal.list()]\n"
    "print([m for m in ('exchange_calendars','holidays','QuantLib','workalendar') if m in sys.modules])\n"
)
print(subprocess.run([sys.executable, "-c", code], capture_output=True, text=True).stdout)

[]



## 2. What the snapshot contains

In [4]:
manifest = snapshot.load_manifest()
summary = pd.DataFrame(
    [{"source": e.provider, "version": e.provider_version} for e in manifest.values()]
).value_counts().reset_index(name="calendars")
summary

,source,version,calendars
0,python_holidays,0.83,251
1,quantlib,1.43,91
2,workalendar,17.0.0,76
3,exchange_calendars,4.5.6,59


Every calendar has a manifest row: where it came from, in which version, over what horizon,
and a SHA-256 digest of the file. That digest is what the drift check compares.

In [5]:
entry = manifest["XNYS"]
pd.Series(entry.to_json())

provider                                           exchange_calendars
provider_version                                                4.5.6
upstream                                                         XNYS
bounds                                       [1970-01-01, 2100-12-31]
weekmask                                          Mon Tue Wed Thu Fri
tz                                                   America/New_York
session_start                                                00:00:00
holidays                                                         1227
sha256              0ecf92382cb246f246308d96394af3f2615d417e332076...
dtype: object

The format is **plain text**, one ISO date per line, one file per calendar. That choice is
not incidental: the whole drift mechanism rests on an upstream change arriving as a
*readable pull request*. A Parquet blob would render as "binary file changed"; here the PR
shows the two lines that moved.

In [6]:
path = snapshot.DATA_DIR / "calendars" / entry.filename
print(f"{path.name} — {entry.holidays} lines\n")
print("\n".join(path.read_text().splitlines()[:6]))
print("…")

XNYS.csv — 1227 lines

1970-01-01
1970-02-23
1970-03-27
1970-07-03
1970-09-07
1970-11-26
…


## 3. Honest bounds

Bounds reflect what the source can **actually** answer, not what it was asked for. Two
distinct causes:

- the upstream refuses outright (Tokyo before 1997, Hong Kong after 2049);
- the upstream *degrades silently* — lunar, Hebrew and Islamic holidays are tabulated, and
  the tables end without saying so.

In [7]:
from datetime import date

clipped = [(n, e) for n, e in manifest.items() if e.bounds[1] != date(2100, 12, 31)]
print(f"{len(clipped)} calendars out of {len(manifest)} have a narrowed horizon\n")
pd.DataFrame(
    [{"calendar": n, "source": e.provider, "from": e.bounds[0], "to": e.bounds[1]}
     for n, e in sorted(clipped)[:12]]
).set_index("calendar")

62 calendars out of 477 have a narrowed horizon



,source,from,to
calendar,,,
XBOM,exchange_calendars,1997-01-01,2024-12-31
XHKG,exchange_calendars,1970-01-01,2049-12-31
XKRX,exchange_calendars,1970-01-01,2050-12-31
XSAU,exchange_calendars,2021-01-01,2024-12-31
XSES,exchange_calendars,1986-01-01,2024-12-31
XSHG,exchange_calendars,1990-12-03,2025-12-31
country:AE,python_holidays,1972-01-01,2077-12-31
country:BH,python_holidays,1972-01-01,2076-12-31
country:BN,python_holidays,1984-01-01,2077-12-31


The second case is the nastier one. Past 2026, QuantLib's Shanghai calendar returns **one**
holiday a year instead of eighteen — Chinese New Year has vanished — and keeps answering
"yes, business day" with complete confidence.

Snapshot generation detects that density collapse and clips the horizon, so you get an
error instead of a wrong answer:

In [8]:
china = bcal.get("ql:China.SSE")
print("bounds :", china.bounds)
try:
    china.is_bday("2030-02-14")
except bcal.OutOfBoundsError as exc:
    print("\n" + str(exc))

bounds : (datetime.date(1970, 1, 1), datetime.date(2026, 12, 31))

2030-02-14 is outside the bounds of calendar 'ql:China.SSE' (1970-01-01 to 2026-12-31, inclusive). Rebuild the calendar with wider `bounds`, or raise MAX_YEAR in better_calendar.core.epoch.


## 4. How an upstream correction reaches you

```
exchange-calendars publishes 4.6.0
        ↓
weekly CI job: installs the latest versions,
regenerates in memory, compares against the committed files
        ↓
difference found → non-zero exit → a pull request is opened
        ↓
    you read the PR:   XNYS
                       + 2027-05-31
                       - 2027-06-01
        ↓
you merge (or not) → a new release of better-calendar
```

The key point: the change is a pull request somebody **reads and approves**, not a side
effect of a `pip upgrade`.

In [9]:
# The command line that drives all of this.
for command in (
    ["describe", "rate:SOFR"],
    ["next", "XNYS", "2026-07-31", "+5"],
    ["list", "--provider", "builtin"],
):
    out = subprocess.run(
        [sys.executable, "-m", "better_calendar.cli", *command],
        capture_output=True, text=True,
    )
    print(f"$ better-calendar {' '.join(command)}")
    print("\n".join(out.stdout.splitlines()[:8]))
    print()

$ better-calendar describe rate:SOFR
{
  "name": "rate:SOFR",
  "weekmask": "Mon Tue Wed Thu Fri",
  "bounds": [
    "1970-01-01",
    "2100-12-31"
  ],
  "tz": null,

$ better-calendar next XNYS 2026-07-31 +5
2026-08-07



$ better-calendar list --provider builtin
crypto:24x7
weekday



`better-calendar diff` regenerates everything in memory and compares. Non-zero exit if any
date moved. That is what the weekly job runs — here on three calendars so it stays quick
(it needs the provider extras installed).

In [10]:
out = subprocess.run(
    [sys.executable, "-m", "better_calendar.cli", "diff",
     "--provider", "quantlib", "--only", "fin:TARGET2,rate:SOFR,fin:LNB"],
    capture_output=True, text=True,
)
print("exit code :", out.returncode)
print(out.stdout or out.stderr.splitlines()[-1])

exit code : 0
snapshot is current (3 calendars, versions {'quantlib': '1.43'})



## 5. Your own calendars

A desk closes on 24 December; the euro area does not. That is a local fact, not a QuantLib
error — and the answer is **never** to fork a provider calendar, because a fork quietly
stops receiving upstream corrections.

### Option A — the configuration file

Found through `$BETTER_CALENDAR_CONFIG`, or `./better-calendar.yaml` in the working
directory. TOML works identically.

In [11]:
import os
import tempfile

from better_calendar.calendars.registry import reload_config

CONFIG = """
calendars:
  desk:paris:
    base: fin:TARGET2
    extra_holidays: ["2026-12-24", "2026-12-31"]
    tz: Europe/Paris

  desk:gulf:
    weekmask: "Sun Mon Tue Wed Thu"
    bounds: ["2020-01-01", "2035-12-31"]
    tz: Asia/Dubai
"""

folder = Path(tempfile.mkdtemp())
(folder / "better-calendar.yaml").write_text(CONFIG)
os.environ["BETTER_CALENDAR_CONFIG"] = str(folder / "better-calendar.yaml")
reload_config()

desk = bcal.get("desk:paris")
base = bcal.get("fin:TARGET2")
pd.DataFrame(
    [
        {"date": d, "fin:TARGET2": base.is_bday(d), "desk:paris": desk.is_bday(d)}
        for d in ("2026-04-06", "2026-12-24", "2026-12-25", "2026-12-31")
    ]
).set_index("date")

,fin:TARGET2,desk:paris
date,,
2026-04-06,False,False
2026-12-24,True,False
2026-12-25,False,False
2026-12-31,True,False


The desk inherits **everything** TARGET2 closes (Easter Monday), plus its own days. And the
base is untouched.

In [12]:
print("desk:gulf weekmask :", bcal.get("desk:gulf").weekmask)
print("Sunday a business day?", bcal.get("desk:gulf").is_bday("2026-08-02"))
print("provenance         :", bcal.describe("desk:paris")["provider"],
      "/", bcal.describe("desk:paris")["provider_version"])

desk:gulf weekmask : Mon Tue Wed Thu Sun
Sunday a business day? True
provenance         : custom / fin:TARGET2


Naming an entry after a shipped calendar **shadows** it, so existing call sites pick up the
local version with no code change:

In [13]:
(folder / "better-calendar.yaml").write_text("""
calendars:
  XNYS:
    base: XNYS
    extra_holidays: ["2026-11-27"]
""")
reload_config()

print("day after Thanksgiving a NYSE business day?", bcal.get("XNYS").is_bday("2026-11-27"))
print("through the NYSE alias too              ?", bcal.get("NYSE").is_bday("2026-11-27"))

day after Thanksgiving a NYSE business day? False
through the NYSE alias too              ? False


In [14]:
del os.environ["BETTER_CALENDAR_CONFIG"]
reload_config()
print("config removed, XNYS returns to the snapshot :", bcal.get("XNYS").is_bday("2026-11-27"))

config removed, XNYS returns to the snapshot : True


### Option B — programmatically

In [15]:
from better_calendar import Calendar

custom = bcal.get("XNYS").with_holidays(["2026-11-27"], name="desk:us")
bcal.register("desk:us", custom)

print("desk:us  :", bcal.get("desk:us").is_bday("2026-11-27"))
print("XNYS     :", bcal.get("XNYS").is_bday("2026-11-27"))
print("listed   :", "desk:us" in bcal.list())

bcal.unregister("desk:us")

desk:us  : False
XNYS     : True
listed   : True


### Resolution order

1. already a `Calendar` — passes through untouched;
2. registered via `register()`;
3. defined in the configuration file;
4. built in (`weekday`, `crypto:24x7`);
5. the committed snapshot;
6. an alias, then start again at step 2.

## Recap

| Call / command | Role |
|---|---|
| `snapshot.load_manifest()` | the index: source, version, bounds, digest |
| `bcal.describe(name)` | the same for one calendar |
| `better-calendar snapshot` | regenerate from the upstream sources |
| `better-calendar diff` | detect drift, non-zero exit |
| `better-calendar describe` / `next` / `list` | command-line inspection |
| `./better-calendar.yaml` | organisation calendars, composed |
| `register()` / `unregister()` | the same, programmatically |

**Back to the start:** [01 — Getting started](01-getting-started.ipynb)